In [1]:
!python -m pip install -U python-woc pandas matplotlib
# please clear the output of this cell in your notebook before checking it in

Defaulting to user installation because normal site-packages is not writeable


In [2]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
# creates the client
woc = WocMapsRemote( base_url="https://worldofcode.org/api/")
# if you got an API key
# woc = WocMapsRemote( base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY" )

# Now for each of the ten projects you were assigned get commits, e.g.
projects = ['nwfsc-cb/OCNMS', 'skuschel/postpic', 'florianhartig/DHARMa', 'Acellera/moleculekit', 'druid-io/pydruid', 'projectchrono/chrono',
            'MolSSI/QCEngine', 'Radiomics/pyradiomics', 'jalammar/ecco', 'pyscf/pyscf',]
list_df_commits = []
for prj in projects:
  # Convert GitHub repository names to WoC V2412 project names
  prj = prj.lower().replace('/','_',2)
  commits = woc.get_values('p2c', prj)
  df = pd.DataFrame(commits, columns=["sha1"])
  df['project'] = prj
  list_df_commits.append(df)
df = pd.concat(list_df_commits)
#perhaps save the list (if no errors) so you do not need to retrieve them again
df.to_csv('df_commits.csv')
df.head(1)

,sha1,project
0,00a981cbd61fe7e318a3afee58ab02787f864990,nwfsc-cb_ocnms


In [3]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time
# let us first split df['sha1'] in chunks
chunks = [df['sha1'][x:x+10] for x in range(0, len(df), 10)]
commit_data = []

for chunk in tqdm(chunks): # iterate over the commits
  # res, err = woc.show_content_many('commit',chunk.to_list())

  # commit.tch returns the same data but faster
  res, err = woc.get_values_many('commit.tch',chunk.to_list())
  res = {k: v[0] for k, v in res.items()}  # this conversion is necessary because of the internal implementation

  # to walk around the rate limit
  time.sleep(1)

  if err: # check for errors
    print('Got Errors', err)

  for commit_sha, commit in res.items():
    # flatten commit objects
    commit_data.append({
          'commit': commit_sha,
          'tree': commit[0],
          'parent': list(commit[1]),
          'author': commit[2][0],
          'author_time': int(commit[2][1]),
          'author_tz': commit[2][2],
          'committer': commit[3][0],
          'committer_time': int(commit[3][1]),
          'committer_tz': commit[3][2],
          'message': commit[4],
    })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
df_commit_data.head(2)

  1%|▉                                                                             | 56/4785 [00:57<1:20:55,  1.03s/it]

Got Errors {'0498ba3fc38386c134ead3a3eb89d984d7ba2ae0': 'Key 0498ba3fc38386c134ead3a3eb89d984d7ba2ae0 not found in /da5_fast/All.sha1c/commit_4.tch'}


  2%|█▎                                                                            | 82/4785 [01:24<1:20:20,  1.02s/it]

Got Errors {'395e21365b03819bea47d53ac23cf49ccb2d58e6': 'Key 395e21365b03819bea47d53ac23cf49ccb2d58e6 not found in /da5_fast/All.sha1c/commit_57.tch'}


  2%|█▊                                                                           | 110/4785 [01:52<1:19:39,  1.02s/it]

...

Got Errors {'e78766d6084b1c3955423b4a39463eaadfe0887c': 'Key e78766d6084b1c3955423b4a39463eaadfe0887c not found in /da5_fast/All.sha1c/commit_103.tch'}


 98%|██████████████████████████████████████████████████████████████████████████▎ | 4680/4785 [1:20:35<01:49,  1.04s/it]

Got Errors {'ee7e5a8f48a8182c6966d54fee36f6043b816a24': 'Key ee7e5a8f48a8182c6966d54fee36f6043b816a24 not found in /da5_fast/All.sha1c/commit_110.tch'}


100%|████████████████████████████████████████████████████████████████████████████| 4785/4785 [1:22:25<00:00,  1.03s/it]


,commit,tree,parent,author,author_time,author_tz,committer,committer_time,committer_tz,message,sha1,project
0,00a981cbd61fe7e318a3afee58ab02787f864990,f2d8ee98870aeeb9e085439e03c5e2f87f781c6a,[6d78feb8e94f93bdf2efb1342f0c98a68b715a8d],Nick Tolimieri <nick.tolimieri@noaa.gov>,1729704027,-0700,Nick Tolimieri <nick.tolimieri@noaa.gov>,1729704027,-0700,Finished initial complete update.\n\nFiles sho...,00a981cbd61fe7e318a3afee58ab02787f864990,nwfsc-cb_ocnms
1,00e5601c2cb1e582978925ef2cdd2521e05eb875,c9b1cb9f53a5614a9c1d5264d97681b62bde952b,[68ffb5d27c84e50b173eb336b13953fda2cf3a4e],jameals <jameal.samhouri@noaa.gov>,1513280110,-0800,jameals <jameal.samhouri@noaa.gov>,1513280110,-0800,updated ms\n,00e5601c2cb1e582978925ef2cdd2521e05eb875,nwfsc-cb_ocnms


In [4]:
# what does the first record looks like?
# commit  parents
v = commit_data[0]
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
print(v)

{'commit': '00a981cbd61fe7e318a3afee58ab02787f864990', 'tree': 'f2d8ee98870aeeb9e085439e03c5e2f87f781c6a', 'parent': ['6d78feb8e94f93bdf2efb1342f0c98a68b715a8d'], 'author': 'Nick Tolimieri <nick.tolimieri@noaa.gov>', 'author_time': 1729704027, 'author_tz': '-0700', 'committer': 'Nick Tolimieri <nick.tolimieri@noaa.gov>', 'committer_time': 1729704027, 'committer_tz': '-0700', 'message': 'Finished initial complete update.\n\nFiles should now run annual update. Have not updated the multivariate file yet.\n'}


In [5]:
# Now that the data is retrieved, save it and commit to your GH fork
# First, we need to flatten info in order to export as csv
# our dataframe will have columns project,'commit, author, time, message'
dfinf = pd.DataFrame({
    'project':  df_commit_data['project'],
    'commit':   df_commit_data['commit'],
    'author':   df_commit_data['author'],
    'time':     df_commit_data['author_time'],
    'message':  df_commit_data['message'],
})


In [6]:
# check if it has the right content
dfinf.head(1)

,project,commit,author,time,message
0,nwfsc-cb_ocnms,00a981cbd61fe7e318a3afee58ab02787f864990,Nick Tolimieri <nick.tolimieri@noaa.gov>,1729704027,Finished initial complete update.\n\nFiles sho...


In [7]:
#mode a means append, so you have all your projects in the same file
yournetid='oyoungbl'
dfinf.to_csv(yournetid+'_project_summary.csv', index=False,sep=';', mode='a', header=False)

# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github

In [8]:
df_commit_data['dt'] = pd.to_datetime(df_commit_data['author_time'], unit='s')
woc_stats = df_commit_data.groupby('project').agg(
    commits=('commit', 'nunique'),
    authors=('author', 'nunique'),
    max_time=('dt', 'max'),
    min_time=('dt', 'min'))
woc_stats

,commits,authors,max_time,min_time
project,,,,
acellera_moleculekit,1477,19,2025-10-20 11:34:40,2019-03-28 10:55:31
druid-io_pydruid,891,200,2025-10-09 19:32:30,2013-08-28 00:52:10
florianhartig_dharma,1281,45,2025-10-22 12:01:59,2015-02-10 15:43:53
jalammar_ecco,349,31,2025-09-29 19:20:49,2020-11-07 10:03:22
molssi_qcengine,2868,98,2025-11-05 16:40:37,2018-03-01 22:25:36
nwfsc-cb_ocnms,532,13,2025-08-14 20:32:09,2015-11-16 20:53:30
projectchrono_chrono,23025,266,2025-11-05 14:05:06,2010-02-23 11:03:14
pyscf_pyscf,14445,356,2025-11-05 21:06:40,2014-05-02 18:42:26
radiomics_pyradiomics,1679,58,2025-10-27 22:04:30,2015-09-15 15:19:31


In [9]:
import requests
gh_projects = projects
headers = {}  # optionally {"Authorization": "token YOUR_TOKEN"}
rows = []
for p in gh_projects:
    r = requests.get(f"https://api.github.com/repos/{p}", headers=headers).json()
    c = requests.get(f"https://api.github.com/repos/{p}/commits?per_page=1", headers=headers).json()
    rows.append({'project': p,
                 'stars': r.get('stargazers_count'),
                 'forks': r.get('forks_count'),
                 'last_commit': c[0]['commit']['committer']['date'] if isinstance(c, list) and c else None})
gh_stats = pd.DataFrame(rows)
gh_stats

,project,stars,forks,last_commit
0,nwfsc-cb/OCNMS,1,0,2026-09-25T18:45:26Z
1,skuschel/postpic,69,27,2026-06-08T13:25:46Z
2,florianhartig/DHARMa,265,31,2026-09-25T15:06:13Z
3,Acellera/moleculekit,246,34,2026-09-28T13:33:29Z
4,druid-io/pydruid,520,204,2026-05-04T23:43:54Z
5,projectchrono/chrono,3057,637,2026-09-26T17:45:48Z
6,MolSSI/QCEngine,211,92,2026-08-18T20:32:51Z
7,Radiomics/pyradiomics,1459,549,2025-06-17T00:39:11Z
8,jalammar/ecco,2101,175,2024-08-15T19:08:06Z
9,pyscf/pyscf,1681,749,2026-09-28T05:45:43Z


## Results

### GitHub (stars, forks, last commit date)
| project | stars | forks | last commit |
|---|---|---|---|
| nwfsc-cb/OCNMS | 1 | 0 | 2026-09-25T18:45:26Z |
| skuschel/postpic | 69 | 27 | 2026-06-08T13:25:46Z |
| florianhartig/DHARMa | 265 | 31 | 2026-09-25T15:06:13Z |
| Acellera/moleculekit | 246 | 34 | 2026-09-28T13:33:29Z |
| druid-io/pydruid | 520 | 204 | 2026-05-04T23:43:54Z |
| projectchrono/chrono | 3057 | 637 | 2026-09-26T17:45:48Z |
| MolSSI/QCEngine | 211 | 92 | 2026-08-18T20:32:51Z |
| Radiomics/pyradiomics | 1459 | 549 | 2025-06-17T00:39:11Z |
| jalammar/ecco | 2101 | 175 | 2024-08-15T19:08:06Z |
| pyscf/pyscf | 1681 | 749 | 2026-09-28T05:45:43Z |

### World of Code (commits, authors, max time, min time)
| project | commits | authors | max time | min time |
|---|---|---|---|---|
| acellera_moleculekit | 1477 | 19 | 2025-10-20 11:34:40 | 2019-03-28 10:55:31 |
| druid-io_pydruid | 891 | 200 | 2025-10-09 19:32:30 | 2013-08-28 00:52:10 |
| florianhartig_dharma | 1281 | 45 | 2025-10-22 12:01:59 | 2015-02-10 15:43:53 |
| jalammar_ecco | 349 | 31 | 2025-09-29 19:20:49 | 2020-11-07 10:03:22 |
| molssi_qcengine | 2868 | 98 | 2025-11-05 16:40:37 | 2018-03-01 22:25:36 |
| nwfsc-cb_ocnms | 532 | 13 | 2025-08-14 20:32:09 | 2015-11-16 20:53:30 |
| projectchrono_chrono | 23025 | 266 | 2025-11-05 14:05:06 | 2010-02-23 11:03:14 |
| pyscf_pyscf | 14445 | 356 | 2025-11-05 21:06:40 | 2014-05-02 18:42:26 |
| radiomics_pyradiomics | 1679 | 58 | 2025-10-27 22:04:30 | 2015-09-15 15:19:31 |
| skuschel_postpic | 1229 | 21 | 2025-03-18 09:51:30 | 2013-12-05 19:43:54 |